# Computational Complexity Analysis — XGBoost Only

Benchmarks **training time** (one full 10-fold CV round), **inference time** (1000 runs on a random input vector), and **peak memory usage** for a standalone XGBoost classifier.

> mRMR features are loaded from `mrmr_selected_features.json` (no recomputation). Memory stats are tracked via `tracemalloc` and `psutil`.

In [ ]:
%pip install -q openpyxl xgboost imbalanced-learn kagglehub psutil

In [ ]:
import time
import tracemalloc
import json
import os
import numpy as np
import pandas as pd
import psutil
import kagglehub

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier

## 1. Load & Preprocess Data

In [ ]:
path = kagglehub.dataset_download("luzrello/dyslexia")
df_desktop = pd.read_csv(path + "/Dyt-desktop.csv", delimiter=';')

categorical_cols = df_desktop.select_dtypes(include=["object"]).columns
for col in categorical_cols:
    le = LabelEncoder()
    df_desktop[col] = le.fit_transform(df_desktop[col])

duplicates = df_desktop.duplicated().sum()
print(f"Duplicate samples: {duplicates}")
if duplicates > 0:
    df_desktop.drop_duplicates(inplace=True)

X = df_desktop.drop("Dyslexia", axis=1).reset_index(drop=True)
y = df_desktop['Dyslexia'].reset_index(drop=True)

print(f"Dataset shape: {X.shape}  |  Classes: {y.value_counts().to_dict()}")

In [ ]:
classes, samples_in_class = np.unique(y, return_counts=True)
n_classes     = len(classes)
total_samples = len(y)

negative_samples = samples_in_class[0]
positive_samples = samples_in_class[1]
scale_pos_weight = float(negative_samples / positive_samples)

print(f"scale_pos_weight: {scale_pos_weight:.4f}")

## 2. Load mRMR Features from Cache

In [ ]:
MRMR_CACHE = "mrmr_selected_features.json"

if not os.path.exists(MRMR_CACHE):
    raise FileNotFoundError(f"Cache file '{MRMR_CACHE}' not found. Run the mRMR step from Complexity_Analysis.ipynb first.")

with open(MRMR_CACHE) as f:
    selected_features = json.load(f)

print(f"Loaded {len(selected_features)} mRMR-selected features from cache.")
X_sel = X[selected_features]

## 3. Training Time & Memory — One Full 10-Fold Round

In [ ]:
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
process = psutil.Process(os.getpid())

fold_records = []
round_start  = time.perf_counter()

for fold, (train_index, test_index) in enumerate(skf.split(X_sel, y), 1):
    X_train, X_test = X_sel.iloc[train_index], X_sel.iloc[test_index]
    y_train, y_test = y.iloc[train_index],     y.iloc[test_index]

    model = XGBClassifier(verbosity=0, scale_pos_weight=scale_pos_weight, random_state=42)

    print(f"Fold {fold:2d} | Training...", flush=True)

    tracemalloc.start()
    rss_before = process.memory_info().rss

    t0_train = time.perf_counter()
    model.fit(X_train, y_train)
    t_train = time.perf_counter() - t0_train

    _, peak_traced = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    rss_after = process.memory_info().rss

    t0_inf = time.perf_counter()
    preds  = model.predict(X_test)
    t_inf  = time.perf_counter() - t0_inf

    acc = accuracy_score(y_test, preds)

    fold_records.append({
        'Fold':                  fold,
        'Training_Time_s':       round(t_train, 4),
        'Fold_Inference_s':      round(t_inf,   6),
        'Test_Samples':          len(y_test),
        'Accuracy':              round(acc, 4),
        'Peak_Traced_MB':        round(peak_traced / 1024**2, 4),
        'RSS_Delta_MB':          round((rss_after - rss_before) / 1024**2, 4),
    })
    print(
        f"Fold {fold:2d} | Train: {t_train:.2f}s | Infer: {t_inf:.4f}s | "
        f"Acc: {acc:.4f} | Peak mem (traced): {peak_traced/1024**2:.2f} MB | "
        f"RSS delta: {(rss_after-rss_before)/1024**2:.2f} MB",
        flush=True
    )

round_total = time.perf_counter() - round_start
print(f"\nTotal wall-clock time for one 10-fold round: {round_total:.2f} s", flush=True)

## 4. Inference Time & Memory — 1000 Runs on a Random Input Vector

In [ ]:
# Train XGBoost on full dataset for inference benchmarking
model_full = XGBClassifier(verbosity=0, scale_pos_weight=scale_pos_weight, random_state=42)

print("Training XGBoost on full dataset...", flush=True)
tracemalloc.start()
rss_before_full = process.memory_info().rss

t0 = time.perf_counter()
model_full.fit(X_sel, y)
full_train_time = time.perf_counter() - t0

_, full_train_peak_traced = tracemalloc.get_traced_memory()
tracemalloc.stop()
rss_after_full = process.memory_info().rss

print(f"Done in {full_train_time:.2f}s", flush=True)
print(f"Full-train peak mem (traced): {full_train_peak_traced/1024**2:.2f} MB")
print(f"Full-train RSS delta: {(rss_after_full - rss_before_full)/1024**2:.2f} MB")

In [ ]:
N_INFERENCE_RUNS = 1000

rng         = np.random.default_rng(seed=0)
feature_min = X_sel.min().values
feature_max = X_sel.max().values

inference_times  = []
inference_mem_mb = []

for i in range(N_INFERENCE_RUNS):
    x_rand = rng.uniform(feature_min, feature_max).reshape(1, -1)
    x_df   = pd.DataFrame(x_rand, columns=X_sel.columns)

    tracemalloc.start()
    t0 = time.perf_counter()
    _  = model_full.predict(x_df)
    inference_times.append(time.perf_counter() - t0)
    _, peak_inf = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    inference_mem_mb.append(peak_inf / 1024**2)

inf_arr     = np.array(inference_times) * 1000   # ms
inf_mem_arr = np.array(inference_mem_mb)          # MB

print(f"Inference over {N_INFERENCE_RUNS} runs (ms):")
print(f"  Mean:   {inf_arr.mean():.4f} ms")
print(f"  Median: {np.median(inf_arr):.4f} ms")
print(f"  Std:    {inf_arr.std():.4f} ms")
print(f"  Min:    {inf_arr.min():.4f} ms")
print(f"  Max:    {inf_arr.max():.4f} ms")
print(f"  P95:    {np.percentile(inf_arr, 95):.4f} ms")
print(f"  P99:    {np.percentile(inf_arr, 99):.4f} ms")
print(f"\nInference peak memory (MB) over {N_INFERENCE_RUNS} runs:")
print(f"  Mean:   {inf_mem_arr.mean():.4f} MB")
print(f"  Median: {np.median(inf_mem_arr):.4f} MB")
print(f"  Max:    {inf_mem_arr.max():.4f} MB")

## 5. Save Results to Excel

In [ ]:
df_folds = pd.DataFrame(fold_records)

summary_row = pd.DataFrame([{
    'Fold':             'MEAN',
    'Training_Time_s':  round(df_folds['Training_Time_s'].mean(), 4),
    'Fold_Inference_s': round(df_folds['Fold_Inference_s'].mean(), 6),
    'Test_Samples':     round(df_folds['Test_Samples'].mean(), 1),
    'Accuracy':         round(df_folds['Accuracy'].mean(), 4),
    'Peak_Traced_MB':   round(df_folds['Peak_Traced_MB'].mean(), 4),
    'RSS_Delta_MB':     round(df_folds['RSS_Delta_MB'].mean(), 4),
}])
total_row = pd.DataFrame([{
    'Fold':             'TOTAL',
    'Training_Time_s':  round(df_folds['Training_Time_s'].sum(), 4),
    'Fold_Inference_s': round(df_folds['Fold_Inference_s'].sum(), 6),
    'Test_Samples':     df_folds['Test_Samples'].sum(),
    'Accuracy':         '',
    'Peak_Traced_MB':   '',
    'RSS_Delta_MB':     '',
}])
df_training = pd.concat([df_folds, summary_row, total_row], ignore_index=True)

df_inference_all = pd.DataFrame({
    'Run':               range(1, N_INFERENCE_RUNS + 1),
    'Inference_Time_ms': np.round(inf_arr, 6),
    'Peak_Mem_MB':       np.round(inf_mem_arr, 6),
})

df_summary = pd.DataFrame([
    {'Metric': 'mRMR features loaded from cache',              'Value': len(selected_features)},
    {'Metric': 'Full-train time (s)',                          'Value': round(full_train_time, 4)},
    {'Metric': 'Full-train peak mem traced (MB)',              'Value': round(full_train_peak_traced/1024**2, 4)},
    {'Metric': 'Full-train RSS delta (MB)',                    'Value': round((rss_after_full - rss_before_full)/1024**2, 4)},
    {'Metric': 'Mean inference (ms)',                          'Value': round(inf_arr.mean(), 4)},
    {'Metric': 'Median inference (ms)',                        'Value': round(np.median(inf_arr), 4)},
    {'Metric': 'Std inference (ms)',                           'Value': round(inf_arr.std(), 4)},
    {'Metric': 'Min inference (ms)',                           'Value': round(inf_arr.min(), 4)},
    {'Metric': 'Max inference (ms)',                           'Value': round(inf_arr.max(), 4)},
    {'Metric': 'P95 inference (ms)',                           'Value': round(np.percentile(inf_arr, 95), 4)},
    {'Metric': 'P99 inference (ms)',                           'Value': round(np.percentile(inf_arr, 99), 4)},
    {'Metric': 'Mean inference peak mem (MB)',                 'Value': round(inf_mem_arr.mean(), 4)},
    {'Metric': 'Median inference peak mem (MB)',               'Value': round(np.median(inf_mem_arr), 4)},
    {'Metric': 'Max inference peak mem (MB)',                  'Value': round(inf_mem_arr.max(), 4)},
    {'Metric': 'N inference runs',                             'Value': N_INFERENCE_RUNS},
])

excel_path = "../../Plots & Visuals/Complexity_Analysis_XGB.xlsx"
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    df_training.to_excel(writer,      index=False, sheet_name='Training_Time')
    df_inference_all.to_excel(writer, index=False, sheet_name='Inference_Time_All_Runs')
    df_summary.to_excel(writer,       index=False, sheet_name='Summary')

print(f"Saved to: {excel_path}")